##Step 1 — Install Required Libraries

In [1]:
!pip install yfinance plotly transformers torch kaleido reportlab -q

##Step 2 — Import Libraries

In [2]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px

from datetime import datetime, timedelta

import yfinance as yf

from transformers import pipeline

import warnings
warnings.filterwarnings("ignore")

##Step 4 — Upload Your CSV File

In [3]:
from google.colab import files

raw_df = pd.read_csv('/content/portfolio_data.csv')

##Step 5 — Load the Dataset

In [4]:
input_file = 'portfolio_data.csv'

raw_df = pd.read_csv(input_file)

raw_df.head()

,portfolio_id,stock_symbol,stock_name,quantity,stock_sector,purchase_price
0,PORT-1001,KO,The Coca-Cola Company,140,Consumer Defensive,57.1340
1,PORT-1001,WMT,Walmart Inc.,9,Consumer Defensive,100.7590
2,PORT-1001,MSFT,Microsoft Corporation,68,Technology,326.0093
3,PORT-1001,PLTR,Palantir Technologies Inc.,6,Technology,139.1355
4,PORT-1001,GE,General Electric Company,32,Industrials,231.5633


##Step 6 — Data Quality Checks

In [5]:
print("Shape:", raw_df.shape)

print("\nMissing Values:")
print(raw_df.isna().sum())

print("\nDuplicate Rows:")
print(raw_df.duplicated().sum())

Shape: (2000, 6)

Missing Values:
portfolio_id      0
stock_symbol      0
stock_name        0
quantity          0
stock_sector      0
purchase_price    0
dtype: int64

Duplicate Rows:
2


##Step 7 — Clean the Dataset

In [6]:
clean_df = raw_df.drop_duplicates().copy()

print(clean_df.shape)

(1998, 6)


##Step 8 — Extract Unique Stock Tickers

In [7]:
tickers = clean_df['stock_symbol'].dropna().unique().tolist()

print(tickers)
print("Total Unique Stocks:", len(tickers))

['KO', 'WMT', 'MSFT', 'PLTR', 'GE', 'MA', 'QQQ', 'PG', 'ABBV', 'INTC', 'GOOG', 'VOO', 'BA', 'NVDA', 'TSLA', 'PYPL', 'JPM', 'CVS', 'SOFI', 'AAPL', 'UNH', 'AMZN', 'XOM']
Total Unique Stocks: 23


Step 9 — Define Time Periods

In [8]:
current_year = datetime.now().year

ytd_start = pd.Timestamp(f"{current_year}-01-01")

three_years_ago = pd.Timestamp(
    datetime.now() - timedelta(days=3*365)
)

five_years_ago = pd.Timestamp(
    datetime.now() - timedelta(days=5*365)
)

##Step 10 — Download Market Data

In [9]:
prices = yf.download(
    tickers,
    start=five_years_ago.strftime("%Y-%m-%d"),
    auto_adjust=True,
    progress=False
)['Close']

prices.head()

Ticker,AAPL,ABBV,AMZN,BA,CVS,GE,GOOG,INTC,JPM,KO,...,PG,PLTR,PYPL,QQQ,SOFI,TSLA,UNH,VOO,WMT,XOM
Date,,,,,,,,,,,,,,,,,,,,,
2021-06-01,121.142151,93.024612,160.932495,254.729996,72.756325,68.891762,120.501190,52.060863,146.340363,47.628403,...,116.998421,23.059999,257.891174,322.978546,22.650000,207.966660,373.318665,359.264923,44.300377,50.306797
2021-06-02,121.902473,92.353111,161.699493,255.619995,72.857986,68.599693,120.078178,52.600780,146.349182,47.817947,...,117.509117,24.450001,260.775787,323.609344,23.209999,201.706665,372.520172,359.870239,44.203438,50.706184
2021-06-03,120.420837,93.024612,159.350494,250.320007,73.764557,68.599693,119.251457,51.466038,146.446091,47.938568,...,119.613441,23.629999,256.419067,320.241943,22.680000,190.946671,374.456909,358.501373,44.281620,50.905872
2021-06-04,122.711525,93.148964,160.311005,249.919998,73.264687,67.966721,121.589760,52.500114,146.684097,48.455524,...,119.701492,24.030001,261.641144,325.676422,20.820000,199.683334,372.327393,361.751038,44.359791,51.130535
2021-06-07,122.721252,93.687843,159.900497,252.660004,73.146057,67.723320,122.300415,52.243885,145.996689,48.283215,...,120.115334,24.459999,259.214142,326.646759,21.209999,201.710007,367.380035,361.443848,44.062706,50.797714


##Step 11 — Build Market Metrics

In [10]:
market_data = {}

for ticker in tickers:

    try:

        s = prices[ticker].dropna()

        if s.empty:
            raise ValueError("No data")

        current_price = s.iloc[-1]

        def perf_since(start_date):

            period = s.loc[start_date:]

            if period.empty:
                return None

            return ((current_price / period.iloc[0]) - 1) * 100

        market_data[ticker] = {

            "current_price": round(current_price, 2),

            "YTD_Perf": round(perf_since(ytd_start) or 0, 2),

            "3YR_Perf": round(perf_since(three_years_ago) or 0, 2),

            "5YR_Perf": round(
                ((current_price / s.iloc[0]) - 1) * 100,
                2
            )
        }

    except Exception as e:

        print(f"{ticker} failed: {e}")

Step 12 — Merge Market Data into Main Dataset

In [11]:
market_df = pd.DataFrame.from_dict(
    market_data,
    orient='index'
).reset_index()

market_df.rename(
    columns={'index':'stock_symbol'},
    inplace=True
)

clean_df = clean_df.merge(
    market_df,
    on='stock_symbol',
    how='left'
)

clean_df.head()

,portfolio_id,stock_symbol,stock_name,quantity,stock_sector,purchase_price,current_price,YTD_Perf,3YR_Perf,5YR_Perf
0,PORT-1001,KO,The Coca-Cola Company,140,Consumer Defensive,57.1340,80.41,17.13,47.39,68.83
1,PORT-1001,WMT,Walmart Inc.,9,Consumer Defensive,100.7590,118.90,5.86,150.93,168.40
2,PORT-1001,MSFT,Microsoft Corporation,68,Technology,326.0093,426.99,-9.31,33.10,79.84
3,PORT-1001,PLTR,Palantir Technologies Inc.,6,Technology,139.1355,143.34,-14.61,874.44,521.60
4,PORT-1001,GE,General Electric Company,32,Industrials,231.5633,320.82,0.17,302.24,365.69


##Step 13 — Calculate Portfolio Metrics

In [12]:
clean_df['total_cost'] = (
    clean_df['purchase_price'] *
    clean_df['quantity']
)

clean_df['current_value'] = (
    clean_df['current_price'] *
    clean_df['quantity']
)

portfolio_totals = clean_df.groupby(
    'portfolio_id'
)['current_value'].transform('sum')

clean_df['percent_of_portfolio'] = round(
    (clean_df['current_value'] / portfolio_totals) * 100,
    2
)

clean_df.head()

,portfolio_id,stock_symbol,stock_name,quantity,stock_sector,purchase_price,current_price,YTD_Perf,3YR_Perf,5YR_Perf,total_cost,current_value,percent_of_portfolio
0,PORT-1001,KO,The Coca-Cola Company,140,Consumer Defensive,57.1340,80.41,17.13,47.39,68.83,7998.7600,11257.40,0.44
1,PORT-1001,WMT,Walmart Inc.,9,Consumer Defensive,100.7590,118.90,5.86,150.93,168.40,906.8310,1070.10,0.04
2,PORT-1001,MSFT,Microsoft Corporation,68,Technology,326.0093,426.99,-9.31,33.10,79.84,22168.6324,29035.32,1.13
3,PORT-1001,PLTR,Palantir Technologies Inc.,6,Technology,139.1355,143.34,-14.61,874.44,521.60,834.8130,860.04,0.03
4,PORT-1001,GE,General Electric Company,32,Industrials,231.5633,320.82,0.17,302.24,365.69,7410.0256,10266.24,0.40


##Step 14 — Donut Chart

In [13]:
stock_allocation = clean_df.groupby(
    'stock_name'
).agg(
    current_allocation=('current_value', 'sum')
).reset_index()

total_value = stock_allocation[
    'current_allocation'
].sum()

stock_allocation['allocation_%'] = round(
    (
        stock_allocation['current_allocation']
        / total_value
    ) * 100,
    2
)

fig = go.Figure(
    data=[
        go.Pie(
            labels=stock_allocation['stock_name'],
            values=stock_allocation['allocation_%'],
            hole=0.4
        )
    ]
)

fig.update_layout(
    title='Portfolio Allocation'
)

fig.show()

##Step 15 — Executive Summary

In [14]:
total_invested = clean_df['total_cost'].sum()

total_current = clean_df['current_value'].sum()

profit_loss = total_current - total_invested

roi = (profit_loss / total_invested) * 100

print("----- EXECUTIVE SUMMARY -----")

print(f"AUM: ${total_current:,.2f}")

print(f"Invested: ${total_invested:,.2f}")

print(f"P&L: ${profit_loss:,.2f}")

print(f"ROI: {roi:.2f}%")

----- EXECUTIVE SUMMARY -----
AUM: $42,103,101.22
Invested: $37,451,324.71
P&L: $4,651,776.51
ROI: 12.42%


##Step 16 — Generate AI Summary

In [15]:
generator = pipeline(
    "text-generation",
    model="distilgpt2"
)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

In [16]:
prompt = f"""
Generate a professional executive summary
for an investment portfolio.

ROI is {roi:.2f}%.
Total portfolio value is ${total_current:,.2f}.

Discuss performance and outlook.
"""

result = generator(
    prompt,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7
)

print(result[0]['generated_text'])

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_th


Generate a professional executive summary
for an investment portfolio.

ROI is 12.42%.
Total portfolio value is $42,103,101.22.

Discuss performance and outlook.
For more information about ROI, visit www.roi.gov/
This article is part of a series on the subject of ROI. For more information about the ROI, visit www.roi.gov/
The U.S. government has been conducting a national audit of the U.S. government and its performance, and has been conducting a national audit of the U.S. government and its performance, and has been conducting a national audit of the U.S


In [17]:
portfolio_summary_text = f"""
Executive Portfolio Summary

The investment portfolio achieved a total ROI of {roi:.2f}%,
with total assets under management valued at
${total_current:,.2f}.

Overall portfolio performance remained positive,
supported by diversified stock holdings and
strong market performance across key sectors.

The portfolio demonstrates stable growth potential
with continued opportunities for long-term value appreciation.
"""

print(portfolio_summary_text)


Executive Portfolio Summary

The investment portfolio achieved a total ROI of 12.42%,
with total assets under management valued at
$42,103,101.22.

Overall portfolio performance remained positive,
supported by diversified stock holdings and
strong market performance across key sectors.

The portfolio demonstrates stable growth potential
with continued opportunities for long-term value appreciation.



##Step 17 — Save Final Dataset

In [18]:
clean_df.to_csv(
    "portfolio_data_final.csv",
    index=False
)

##Step 18 — Download Final CSV

In [19]:
files.download("portfolio_data_final.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##Step 19 — Export Charts as Images

In [20]:
!pip install -U kaleido plotly -q
import plotly.graph_objects as go

In [21]:
fig.write_html("allocation_chart.html")

In [22]:
files.download("allocation_chart.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##Step 22 — Import PDF Libraries

In [23]:
!pip install reportlab -q

In [24]:
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Image,
    Table,
    TableStyle
)

from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import letter

##Step 23 — Save Chart as PNG

In [25]:
fig.write_html("allocation_chart.html")

##Step 24 — Create PNG Chart Using Matplotlib

In [26]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,8))

plt.pie(
    stock_allocation['allocation_%'],
    labels=stock_allocation['stock_name'],
    autopct='%1.1f%%'
)

plt.title("Portfolio Allocation")

plt.savefig("allocation_chart.png")

plt.close()

##Step 25 — Build Executive Summary Text

In [27]:
portfolio_summary_text = f"""
Executive Portfolio Summary

The investment portfolio achieved a total ROI of {roi:.2f}%,
with total assets under management valued at
${total_current:,.2f}.

Overall portfolio performance remained positive,
supported by diversified stock holdings and
strong market performance across key sectors.

The portfolio demonstrates stable growth potential
with continued opportunities for long-term value appreciation.
"""

##Step 26 — Create PDF Report

In [28]:
doc = SimpleDocTemplate(
    "Portfolio_Report.pdf",
    pagesize=letter
)

styles = getSampleStyleSheet()

elements = []

##Step 27 — Add Title

In [29]:
title = Paragraph(
    "AI-Powered Portfolio Insights Report",
    styles['Title']
)

elements.append(title)
elements.append(Spacer(1, 20))

##Step 28 — Add KPI Summary

In [30]:
kpi_text = f"""
<b>Total Portfolio Value:</b> ${total_current:,.2f}<br/>
<b>Total ROI:</b> {roi:.2f}%<br/>
<b>Total Invested:</b> ${total_invested:,.2f}<br/>
"""

kpi_paragraph = Paragraph(
    kpi_text,
    styles['BodyText']
)

elements.append(kpi_paragraph)
elements.append(Spacer(1, 20))

##Step 29 — Add Executive Summary

In [31]:
summary = Paragraph(
    portfolio_summary_text,
    styles['BodyText']
)

elements.append(summary)
elements.append(Spacer(1, 20))

##Step 30 — Add Chart Image

In [32]:
chart = Image(
    "allocation_chart.png",
    width=400,
    height=400
)

elements.append(chart)
elements.append(Spacer(1, 20))

##Step 31 — Add Portfolio Summary Table

##remove the below code later

In [33]:
portfolio_summary = clean_df.groupby('portfolio_id').agg(
    unique_assets=('stock_symbol', 'nunique'),
    total_cost=('total_cost', 'sum'),
    current_value=('current_value', 'sum'),
).reset_index()

portfolio_summary['P&L_($)'] = (
    portfolio_summary['current_value']
    - portfolio_summary['total_cost']
)

portfolio_summary['Return_(%)'] = round(
    (
        portfolio_summary['P&L_($)']
        / portfolio_summary['total_cost']
    ) * 100,
    2
)

portfolio_summary.head()

,portfolio_id,unique_assets,total_cost,current_value,P&L_($),Return_(%)
0,PORT-1001,23,2.218889e+06,2580711.29,361822.6098,16.31
1,PORT-1002,23,2.528419e+06,2823463.71,295044.6492,11.67
2,PORT-1003,23,3.301676e+06,3693436.48,391760.9117,11.87
3,PORT-1004,23,2.082453e+06,2358467.93,276015.0048,13.25
4,PORT-1005,23,2.479255e+06,2727321.45,248066.1564,10.01


In [34]:
table_data = [portfolio_summary.columns.tolist()] + portfolio_summary.values.tolist()

table = Table(table_data)

table.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.grey),
    ('TEXTCOLOR', (0,0), (-1,0), colors.whitesmoke),

    ('ALIGN', (0,0), (-1,-1), 'CENTER'),

    ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),

    ('BOTTOMPADDING', (0,0), (-1,0), 12),

    ('BACKGROUND', (0,1), (-1,-1), colors.beige),

    ('GRID', (0,0), (-1,-1), 1, colors.black),
]))

elements.append(table)

##Step 32 — Generate PDF

In [35]:
doc.build(elements)

print("PDF Report Generated Successfully")

PDF Report Generated Successfully


##Step 33 — Download PDF

In [36]:
from google.colab import files

files.download("Portfolio_Report.pdf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>